# Module 4: Evaluation Metrics — Ejercicios Prácticos

**Objetivos:**
- Calcular y interpretar confusion matrix
- Dominar precision, recall, F1
- Visualizar ROC-AUC y PR curves
- Regresión: R², RMSE, MAE

**Tiempo:** 30 min

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, roc_auc_score,
    r2_score, mean_squared_error, mean_absolute_error
)
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## Ejercicio 1: Confusion Matrix y Métricas Básicas

In [ ]:
# Cargar dataset
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTN: {tn}, FP: {fp}")
print(f"FN: {fn}, TP: {tp}")

# Visualizar
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Ejercicio 2: Precision, Recall, F1

In [ ]:
# Calcular manualmente
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# También con fórmulas manuales
precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual)

print("Métricas de Clasificación:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}  (manual: {precision_manual:.4f})")
print(f"Recall: {recall:.4f}  (manual: {recall_manual:.4f})")
print(f"F1: {f1:.4f}  (manual: {f1_manual:.4f})")

print(f"\n📊 Interpretación:")
print(f"Precision ({precision:.1%}): De los {tp+fp} casos predichos positivos, {tp} fueron reales")
print(f"Recall ({recall:.1%}): De los {tp+fn} casos reales positivos, detecté {tp}")

## Ejercicio 3: ROC-AUC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Plotear
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"Interpretación: Probabilidad que el modelo rank positivo > negativo: {roc_auc:.1%}")

## Ejercicio 4: Precision-Recall Curve

In [ ]:
# PR Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall_vals, precision_vals)

# Plotear
plt.figure(figsize=(10, 8))
plt.plot(recall_vals, precision_vals, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.3f})')
plt.axhline(y=np.mean(y_test), color='red', linestyle='--', lw=2, label=f'Baseline ({np.mean(y_test):.1%})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.show()

print(f"PR-AUC Score: {pr_auc:.4f}")
print(f"Mejor para datasets desbalanceados que ROC-AUC")

## Ejercicio 5: Métricas de Regresión

In [ ]:
# Dataset de regresión
diabetes = load_diabetes()
X_diabetes = diabetes.data
y_diabetes = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# Entrenar modelo
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred_reg = reg_model.predict(X_test)

# Métricas
r2 = r2_score(y_test, y_pred_reg)
mse = mean_squared_error(y_test, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_reg)

print("Métricas de Regresión:")
print(f"R²: {r2:.4f}  (explica {r2*100:.1f}% de la varianza)")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}  (error promedio en unidades originales)")
print(f"MAE: {mae:.4f}   (error promedio absoluto)")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicciones vs Reales
axes[0].scatter(y_test, y_pred_reg, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Valor Real')
axes[0].set_ylabel('Predicción')
axes[0].set_title('Predicciones vs Real')
axes[0].grid(True, alpha=0.3)

# Residuales
residuals = y_test - y_pred_reg
axes[1].scatter(y_pred_reg, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Resumen

In [ ]:
print("="*60)
print("RESUMEN: EVALUATION METRICS")
print("="*60)
print()
print("CLASIFICACIÓN:")
print("  - Balanceado: Accuracy, F1, ROC-AUC")
print("  - Desbalanceado: F1, PR-AUC, Recall")
print()
print("TRADE-OFFS:")
print("  - Precision vs Recall: depende del costo")
print("  - ROC-AUC vs PR-AUC: PR mejor para desbalance")
print()
print("REGRESIÓN:")
print("  - R²: varianza explicada")
print("  - RMSE/MAE: error promedio (unidades originales)")
print()
print("REGLA DE ORO:")
print("  - Define métrica primaria ANTES de entrenar")
print("  - Valida en múltiples splits (cross-validation)")
print("  - Compara train vs test (overfitting?)")
print()
print("="*60)